In [1]:
try:
    import kagglehub
except:
    %pip install kagglehub
    import kagglehub

# Download latest version
path = kagglehub.dataset_download("imbikramsaha/caltech-101")

print("Path to dataset files:", path)

Path to dataset files: /Users/charlottexx/.cache/kagglehub/datasets/imbikramsaha/caltech-101/versions/1


In [2]:
import glob
import os

image_paths = glob.glob(f"{path}/*/*")
if os.path.isdir(image_paths[0]):
    image_paths = glob.glob(f"{path}/*/*/*")

labels = [os.path.basename(os.path.dirname(image_path)) for image_path in image_paths]
print(len(labels), len(set(labels)))

9145 102


In [3]:
import random
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def sample_labels_and_images(
    image_paths,
    labels,
    num_labels=10,
    samples_per_label=10,
    test_size=0.2,
    random_state=42,
):
    random.seed(131266)
    
    unique_labels = list(set(labels))
    selected_unique_labels = random.sample(
        unique_labels, min(num_labels, len(unique_labels))
    )


    label_to_paths = {}
    for path, label in zip(image_paths, labels):
        if label in selected_unique_labels:
            if label not in label_to_paths:
                label_to_paths[label] = []
            label_to_paths[label].append(path)
    
    sampled_image_paths = []
    sampled_labels = []

    for label, paths in label_to_paths.items():
        sample_count = min(samples_per_label, len(paths))
        sampled_paths = random.sample(paths, sample_count)
        sampled_image_paths.extend(sampled_paths)
        sampled_labels.extend([label] * sample_count)

    train_image_paths, test_image_paths, train_labels, test_labels = train_test_split(
        sampled_image_paths,
        sampled_labels,
        test_size=test_size,
        stratify=sampled_labels,
        random_state=random_state,
    )

    return (
        train_image_paths,
        train_labels,
        test_image_paths,
        test_labels,
        selected_unique_labels,
    )



In [4]:
import time
import cv2
import numpy as np
from sklearn.cluster import KMeans
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity


def top_k_accuracy(y_true, y_top_k_predict, k=3):
    correct_predictions = sum(
        1 for true, top_k in zip(y_true, y_top_k_predict) if true in top_k[:k]
    )
    accuracy = correct_predictions / len(y_true) * 100

    return accuracy


def mean_reciprocal_rank(y_true, y_top_k_predict):
    reciprocal_ranks = []
    for true_label, top_k_preds in zip(y_true, y_top_k_predict):
        # Find the rank of the true label in predictions
        try:
            rank = np.where(top_k_preds == true_label)[0][0] + 1
            reciprocal_ranks.append(1 / rank)
        except:
            # True label not found in predictions
            reciprocal_ranks.append(0)

    # Calculate Mean Reciprocal Rank
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr


class CookBook:
    def __init__(
        self, train_paths, train_labels, test_paths, test_labels, n_cluster=100
    ):
        self.train_paths = train_paths
        self.train_labels = np.array(train_labels)
        self.test_paths = test_paths
        self.test_labels = np.array(test_labels)
        self.n_clusters = n_cluster
        self.train_features = None
        self.test_features = None
        self.train()

    def extract_sift_features(self, image_paths):
        if type(image_paths) == str:
            image_paths = [image_paths]

        sift = cv2.SIFT_create()

        # Lists to store descriptors
        all_descriptors = []
        feature_counts = []

        # Extract SIFT features from each image
        for path in tqdm(image_paths, desc="Extracting SIFT Features"):
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

            _, descriptors = sift.detectAndCompute(img, None)

            # If descriptors are found, add them to the list
            if descriptors is not None:
                all_descriptors.append(descriptors)
                feature_counts.append(len(descriptors))

        return all_descriptors, feature_counts

    def train(self):
        start = time.time()
        self.descriptors, _ = self.extract_sift_features(self.train_paths)
        descriptors_stack = np.vstack(self.descriptors)
        print("Kmean clustering...")
        self.kmeans = KMeans(n_clusters=self.n_clusters, init='k-means++', random_state=42, n_init=10)
        self.kmeans.fit(descriptors_stack)

        bow_features = []
        for descriptors in self.descriptors:
            # If no descriptors, return zero vector
            if descriptors is None or len(descriptors) == 0:
                return np.zeros(self.n_clusters)

            # Assign descriptors to nearest visual words
            visual_words = self.kmeans.predict(descriptors)

            # Compute histogram
            histogram, _ = np.histogram(visual_words, bins=range(self.n_clusters + 1))
            bow_features.append(histogram)
        self.train_features = bow_features

        self.tfidf_transformer = TfidfTransformer()
        self.train_features_tfidf = self.tfidf_transformer.fit_transform(
            self.train_features
        )

        self.train_probs = np.array(
            [hist / np.sum(hist) for hist in self.train_features]
        )
        self.train_probs = np.clip(self.train_probs, 1e-10, None)
        print("Total Training Time:", time.time() - start)

    def indexing(self, image_paths):
        if not self.kmeans:
            raise Exception("model haven't train")

        all_descriptors, _ = self.extract_sift_features(image_paths)

        bow_features = []
        for descriptors in all_descriptors:
            # If no descriptors, return zero vector
            if descriptors is None or len(descriptors) == 0:
                return np.zeros(self.n_clusters)

            # Assign descriptors to nearest visual words
            visual_words = self.kmeans.predict(descriptors)

            # Compute histogram
            histogram, _ = np.histogram(visual_words, bins=range(self.n_clusters + 1))
            bow_features.append(histogram)
        return np.array(bow_features)

    def common_word_retrieval(self, query, k=10):
        common_words = np.minimum(query, self.train_features).sum(axis=1)
        top_indices = np.argsort(common_words)[::-1][:k]
        top_labels = self.train_labels[top_indices]

        return top_labels

    def tfidf_retrieval(self, query, k=10):
        query_tfidf = self.tfidf_transformer.transform(query.reshape(1, -1))
        similarities = cosine_similarity(
            query_tfidf, self.train_features_tfidf
        ).flatten()
        top_indices = np.argsort(similarities)[::-1][:k]
        top_labels = self.train_labels[top_indices]

        return top_labels

    def KL_divergence_retrieval(self, query, k=10):
        query_prob = query / np.sum(query)

        query_prob = np.clip(query_prob, 1e-10, None)

        # Compute KL divergence for all training histograms simultaneously
        kl_divergences = np.sum(
            query_prob * np.log(query_prob / self.train_probs), axis=1
        )

        # Get the indices of the top-k smallest KL divergences
        top_indices = np.argsort(kl_divergences)[:k]

        # Retrieve corresponding labels
        top_labels = self.train_labels[top_indices]

        return top_labels

    def evaluate(self, test="common_word_retrieval"):

        if self.test_features is None:
            self.test_features = self.indexing(self.test_paths)
        start = time.time()
        if test == "common_word_retrieval":
            self.test_retrival_results = np.array(
                [self.common_word_retrieval(query) for query in self.test_features]
            )
            self.train_retrival_results = np.array(
                [self.common_word_retrieval(query) for query in self.train_features]
            )

        if test == "tfidf_retrieval":
            self.test_retrival_results = np.array(
                [self.tfidf_retrieval(query) for query in self.test_features]
            )
            self.train_retrival_results = np.array(
                [self.tfidf_retrieval(query) for query in self.train_features]
            )

        if test == "KL_divergence_retrieval":
            self.test_retrival_results = np.array(
                [self.KL_divergence_retrieval(query) for query in self.test_features]
            )
            self.train_retrival_results = np.array(
                [self.KL_divergence_retrieval(query) for query in self.train_features]
            )
        running_time = time.time() - start
        print(test)
        print("n_cluster = ", self.n_clusters)
        print(
            "Train top 3 accuracy:",
            top_k_accuracy(self.train_labels, self.train_retrival_results),
        )
        print(
            "Test top 3 accuracy:",
            top_k_accuracy(self.test_labels, self.test_retrival_results),
        )

        print(
            "Train mean reciprocal rank:",
            mean_reciprocal_rank(self.train_labels, self.train_retrival_results),
        )
        print(
            "Test mean reciprocal rank:",
            mean_reciprocal_rank(self.test_labels, self.test_retrival_results),
        )
        print("Total Runing Time: ", running_time)


In [8]:
# Example usage
train_paths, train_labels, test_paths, test_labels, selected_labels = (
    sample_labels_and_images(
        image_paths, labels, num_labels=5, samples_per_label=20, test_size=0.5
    )
)

# Print out statistics
print("Selected Labels:", selected_labels)
print("\nTraining Set:")
print(f"Total train images: {len(train_paths)}")
from collections import Counter

print("Train samples per label:")
print(Counter(train_labels))

print("\nTesting Set:")
print(f"Total test images: {len(test_paths)}")
print("Test samples per label:")
print(Counter(test_labels))

Selected Labels: ['rooster', 'ewer', 'grand_piano', 'wrench', 'stop_sign']

Training Set:
Total train images: 50
Train samples per label:
Counter({'rooster': 10, 'ewer': 10, 'stop_sign': 10, 'wrench': 10, 'grand_piano': 10})

Testing Set:
Total test images: 50
Test samples per label:
Counter({'wrench': 10, 'grand_piano': 10, 'rooster': 10, 'stop_sign': 10, 'ewer': 10})


In [9]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=50)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 116.64it/s]


Kmean clustering...
Total Training Time: 6.5391480922698975


Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 115.69it/s]

common_word_retrieval
n_cluster =  50
Train top 3 accuracy: 98.0
Test top 3 accuracy: 46.0
Train mean reciprocal rank: 0.9183333333333334
Test mean reciprocal rank: 0.43705555555555553
Total Runing Time:  0.0017747879028320312
tfidf_retrieval
n_cluster =  50
Train top 3 accuracy: 100.0
Test top 3 accuracy: 88.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.776857142857143
Total Runing Time:  0.04104185104370117
KL_divergence_retrieval
n_cluster =  50
Train top 3 accuracy: 100.0
Test top 3 accuracy: 64.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4952142857142857
Total Runing Time:  0.0028688907623291016


In [10]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=100)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 121.09it/s]


Kmean clustering...
Total Training Time: 9.66970705986023


Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 118.58it/s]

common_word_retrieval
n_cluster =  100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 46.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.42979365079365084
Total Runing Time:  0.002256155014038086
tfidf_retrieval
n_cluster =  100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 90.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.7866666666666667
Total Runing Time:  0.04346323013305664
KL_divergence_retrieval
n_cluster =  100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 48.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4139841269841269
Total Runing Time:  0.004443168640136719


In [11]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=200)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 122.50it/s]


Kmean clustering...
Total Training Time: 12.896882057189941


Extracting SIFT Features: 100%|██████████| 50/50 [00:00<00:00, 121.75it/s]


common_word_retrieval
n_cluster =  200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 48.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4558174603174603
Total Runing Time:  0.0026679039001464844
tfidf_retrieval
n_cluster =  200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 92.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.8058333333333333
Total Runing Time:  0.04394698143005371
KL_divergence_retrieval
n_cluster =  200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 54.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.44560317460317456
Total Runing Time:  0.008336067199707031


In [64]:
# Example usage
train_paths, train_labels, test_paths, test_labels, selected_labels = (
    sample_labels_and_images(
        image_paths, labels, num_labels=20, samples_per_label=20, test_size=0.5
    )
)

# Print out statistics
print("Selected Labels:", selected_labels)
print("\nTraining Set:")
print(f"Total train images: {len(train_paths)}")
from collections import Counter

print("Train samples per label:")
print(Counter(train_labels))

print("\nTesting Set:")
print(f"Total test images: {len(test_paths)}")
print("Test samples per label:")
print(Counter(test_labels))

Selected Labels: ['cannon', 'revolver', 'buddha', 'headphone', 'pyramid', 'pagoda', 'flamingo', 'metronome', 'emu', 'watch', 'joshua_tree', 'brain', 'Leopards', 'ewer', 'pigeon', 'garfield', 'wrench', 'dollar_bill', 'anchor', 'nautilus']

Training Set:
Total train images: 200
Train samples per label:
Counter({'anchor': 10, 'ewer': 10, 'headphone': 10, 'revolver': 10, 'pagoda': 10, 'pigeon': 10, 'dollar_bill': 10, 'brain': 10, 'flamingo': 10, 'joshua_tree': 10, 'cannon': 10, 'pyramid': 10, 'nautilus': 10, 'metronome': 10, 'buddha': 10, 'watch': 10, 'Leopards': 10, 'garfield': 10, 'wrench': 10, 'emu': 10})

Testing Set:
Total test images: 200
Test samples per label:
Counter({'metronome': 10, 'buddha': 10, 'joshua_tree': 10, 'flamingo': 10, 'pigeon': 10, 'pagoda': 10, 'Leopards': 10, 'garfield': 10, 'pyramid': 10, 'dollar_bill': 10, 'emu': 10, 'ewer': 10, 'wrench': 10, 'headphone': 10, 'watch': 10, 'brain': 10, 'cannon': 10, 'revolver': 10, 'anchor': 10, 'nautilus': 10})


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=50)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 162.10it/s]


Kmean clustering...
Total Training Time: 16.45663619041443


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 169.27it/s]


common_word_retrieval
n_cluster =  50
Train top 3 accuracy: 93.0
Test top 3 accuracy: 18.0
Train mean reciprocal rank: 0.8865753968253969
Test mean reciprocal rank: 0.16492460317460317
Total Runing Time:  0.015095949172973633
tfidf_retrieval
n_cluster =  50
Train top 3 accuracy: 100.0
Test top 3 accuracy: 48.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4081269841269842
Total Runing Time:  0.12981915473937988
KL_divergence_retrieval
n_cluster =  50
Train top 3 accuracy: 100.0
Test top 3 accuracy: 36.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.31954960317460335
Total Runing Time:  0.024493932723999023


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=500)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 166.70it/s]


Kmean clustering...
Total Training Time: 145.83734464645386


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 167.42it/s]


common_word_retrieval
n_cluster =  500
Train top 3 accuracy: 100.0
Test top 3 accuracy: 23.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.21301785714285726
Total Runing Time:  0.04225516319274902
tfidf_retrieval
n_cluster =  500
Train top 3 accuracy: 100.0
Test top 3 accuracy: 50.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4382242063492064
Total Runing Time:  0.18481993675231934
KL_divergence_retrieval
n_cluster =  500
Train top 3 accuracy: 100.0
Test top 3 accuracy: 24.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.21253769841269854
Total Runing Time:  0.21191716194152832


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=600)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 163.00it/s]


Kmean clustering...
Total Training Time: 125.42813420295715


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 168.88it/s]


common_word_retrieval
n_cluster =  600
Train top 3 accuracy: 100.0
Test top 3 accuracy: 24.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.22474404761904776
Total Runing Time:  0.04540610313415527
tfidf_retrieval
n_cluster =  600
Train top 3 accuracy: 100.0
Test top 3 accuracy: 50.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4382083333333335
Total Runing Time:  0.1841747760772705
KL_divergence_retrieval
n_cluster =  600
Train top 3 accuracy: 100.0
Test top 3 accuracy: 24.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.212888888888889
Total Runing Time:  0.2571580410003662


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=700)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 163.62it/s]


Kmean clustering...
Total Training Time: 163.38150882720947


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 129.54it/s]


common_word_retrieval
n_cluster =  700
Train top 3 accuracy: 100.0
Test top 3 accuracy: 23.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.21917460317460327
Total Runing Time:  0.0740060806274414
tfidf_retrieval
n_cluster =  700
Train top 3 accuracy: 100.0
Test top 3 accuracy: 49.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4352281746031747
Total Runing Time:  0.26090097427368164
KL_divergence_retrieval
n_cluster =  700
Train top 3 accuracy: 100.0
Test top 3 accuracy: 24.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.21822817460317462
Total Runing Time:  0.43679189682006836


In [66]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=800)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 124.96it/s]


Kmean clustering...
Total Training Time: 219.80080771446228


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 120.98it/s]


common_word_retrieval
n_cluster =  800
Train top 3 accuracy: 100.0
Test top 3 accuracy: 22.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.21943055555555563
Total Runing Time:  0.08175897598266602
tfidf_retrieval
n_cluster =  800
Train top 3 accuracy: 100.0
Test top 3 accuracy: 54.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.44198412698412703
Total Runing Time:  0.26753687858581543
KL_divergence_retrieval
n_cluster =  800
Train top 3 accuracy: 100.0
Test top 3 accuracy: 23.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.2325476190476192
Total Runing Time:  0.4871950149536133


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=900)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 124.92it/s]


Kmean clustering...
Total Training Time: 203.50386095046997


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 128.53it/s]


common_word_retrieval
n_cluster =  900
Train top 3 accuracy: 100.0
Test top 3 accuracy: 23.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.22456150793650814
Total Runing Time:  0.09911584854125977
tfidf_retrieval
n_cluster =  900
Train top 3 accuracy: 100.0
Test top 3 accuracy: 53.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.42558531746031764
Total Runing Time:  0.2743349075317383
KL_divergence_retrieval
n_cluster =  900
Train top 3 accuracy: 100.0
Test top 3 accuracy: 23.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.23349007936507957
Total Runing Time:  0.5609860420227051


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=1000)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 124.72it/s]


Kmean clustering...
Total Training Time: 231.38031697273254


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 132.92it/s]


common_word_retrieval
n_cluster =  1000
Train top 3 accuracy: 100.0
Test top 3 accuracy: 26.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.2360932539682541
Total Runing Time:  0.09432220458984375
tfidf_retrieval
n_cluster =  1000
Train top 3 accuracy: 100.0
Test top 3 accuracy: 50.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.42041468253968245
Total Runing Time:  0.27875208854675293
KL_divergence_retrieval
n_cluster =  1000
Train top 3 accuracy: 100.0
Test top 3 accuracy: 28.499999999999996
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.24205158730158746
Total Runing Time:  0.6057488918304443


In [ ]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=1100)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 117.78it/s]


Kmean clustering...
Total Training Time: 253.8546540737152


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 131.25it/s]


common_word_retrieval
n_cluster =  1100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 26.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.256184523809524
Total Runing Time:  0.13690924644470215
tfidf_retrieval
n_cluster =  1100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 49.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.4437896825396826
Total Runing Time:  0.2791280746459961
KL_divergence_retrieval
n_cluster =  1100
Train top 3 accuracy: 100.0
Test top 3 accuracy: 29.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.24494246031746056
Total Runing Time:  0.7004239559173584


In [69]:
cookbook = CookBook(train_paths, train_labels, test_paths, test_labels, n_cluster=1200)
cookbook.evaluate("common_word_retrieval")
cookbook.evaluate("tfidf_retrieval")
cookbook.evaluate("KL_divergence_retrieval")

Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 129.45it/s]


Kmean clustering...
Total Training Time: 298.25337195396423


Extracting SIFT Features: 100%|██████████| 200/200 [00:01<00:00, 166.42it/s]


common_word_retrieval
n_cluster =  1200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 26.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.24103373015873028
Total Runing Time:  0.10273313522338867
tfidf_retrieval
n_cluster =  1200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 48.0
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.41122619047619047
Total Runing Time:  0.19833707809448242
KL_divergence_retrieval
n_cluster =  1200
Train top 3 accuracy: 100.0
Test top 3 accuracy: 29.5
Train mean reciprocal rank: 1.0
Test mean reciprocal rank: 0.2488948412698414
Total Runing Time:  0.5235600471496582
